# Validazione out-of-sample per tipo di oggetto

Questo notebook valuta le componenti Petri net associate ai tipi di oggetto `orders`, `items` e `packages` su tracce non utilizzate durante la discovery.

L'obiettivo è verificare la capacità dei modelli di riprodurre comportamento non osservato durante l'addestramento, evitando la condivisione di identificatori di caso o di evento tra training e test.

## Obiettivo e perimetro

La validazione viene eseguita separatamente per ogni tipo di oggetto. Ogni OCEL appiattito viene suddiviso in training e test con un rapporto obiettivo pari a `0.80`.

La Petri net viene scoperta esclusivamente sul training set con Inductive Miner e `noise_threshold = 0.0`. Il test set viene poi utilizzato per calcolare fitness, precisione e generalizzazione.

Questa procedura non costituisce un conformance checking object-centric globalmente sincronizzato.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import pm4py
from pm4py.util import constants

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SOURCE_ROOT = PROJECT_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

constants.SHOW_PROGRESS_BAR = False

from ocpm_partial_order.config import MAIN_DATASET_DB
from ocpm_partial_order.conformance import (
    DEFAULT_NOISE_THRESHOLD,
    DEFAULT_STRUCTURAL_OBJECT_TYPES,
    DEFAULT_TRAIN_RATIO,
    evaluate_structural_holdout,
)
from ocpm_partial_order.io.ocel_loader import load_ocel2_sqlite

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset: {MAIN_DATASET_DB}")



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




Project root: C:\Users\nicol\ocpm-partial-order\ocpm-partial-order
Dataset: C:\Users\nicol\ocpm-partial-order\ocpm-partial-order\data\raw\order_management.sqlite


## Caricamento del log

Il dataset Order Management viene caricato in formato OCEL 2.0. Le analisi successive considerano soltanto i tipi strutturali `orders`, `items` e `packages`.

In [2]:
ocel = load_ocel2_sqlite(MAIN_DATASET_DB)

print("OCEL loaded")
print(f"Events: {len(ocel.events)}")
print(f"Objects: {len(ocel.objects)}")
print(
    "Object types: "
    f"{list(DEFAULT_STRUCTURAL_OBJECT_TYPES)}"
)

OCEL loaded
Events: 21008
Objects: 10825
Object types: ['orders', 'items', 'packages']


## Perché lo split per singolo caso non è sufficiente

Nell'OCEL uno stesso evento può essere associato a più oggetti dello stesso tipo. Separare direttamente gli identificatori degli oggetti può quindi copiare lo stesso evento sia nel training set sia nel test set.

Il controllo seguente simula uno split cronologico ingenuo dei casi e conta gli eventi condivisi.

In [3]:
CASE_COLUMN = "case:concept:name"
EVENT_COLUMN = "ocel:eid"
TIMESTAMP_COLUMN = "time:timestamp"


def naive_shared_event_count(flattened_log, train_ratio):
    case_starts = (
        flattened_log
        .groupby(CASE_COLUMN)[TIMESTAMP_COLUMN]
        .min()
        .sort_values(kind="stable")
    )

    cut = int(len(case_starts) * train_ratio)
    train_cases = set(case_starts.index[:cut].astype(str))
    test_cases = set(case_starts.index[cut:].astype(str))

    case_values = flattened_log[CASE_COLUMN].astype(str)
    train_events = set(
        flattened_log.loc[
            case_values.isin(train_cases),
            EVENT_COLUMN,
        ].astype(str)
    )
    test_events = set(
        flattened_log.loc[
            case_values.isin(test_cases),
            EVENT_COLUMN,
        ].astype(str)
    )

    return len(train_events & test_events)


leakage_rows = []
for object_type in DEFAULT_STRUCTURAL_OBJECT_TYPES:
    flattened_log = pm4py.ocel_flattening(
        ocel,
        object_type,
    )
    leakage_rows.append(
        {
            "object_type": object_type,
            "shared_events_naive_split": (
                naive_shared_event_count(
                    flattened_log,
                    DEFAULT_TRAIN_RATIO,
                )
            ),
        }
    )

leakage_table = pd.DataFrame(leakage_rows)
leakage_table

,object_type,shared_events_naive_split
0,orders,0
1,items,131
2,packages,0


## Suddivisione senza leakage

Per impedire la condivisione di eventi, i casi collegati dallo stesso identificatore di evento vengono riuniti in componenti connesse. Ogni componente viene assegnata interamente al training set oppure al test set.

L'ordinamento temporale delle componenti mantiene il criterio cronologico per quanto possibile, mentre il punto di separazione viene scelto in modo da avvicinarsi al rapporto `0.80`.

In [4]:
evaluations = evaluate_structural_holdout(
    ocel,
    object_types=DEFAULT_STRUCTURAL_OBJECT_TYPES,
    train_ratio=DEFAULT_TRAIN_RATIO,
    noise_threshold=DEFAULT_NOISE_THRESHOLD,
)

split_rows = []
for evaluation in evaluations:
    split = evaluation.split
    split_rows.append(
        {
            "object_type": evaluation.object_type,
            "components": split.component_count,
            "train_components": split.train_component_count,
            "test_components": split.test_component_count,
            "train_cases": split.train_case_count,
            "test_cases": split.test_case_count,
            "train_ratio": split.effective_train_ratio,
            "train_events": split.train_event_count,
            "test_events": split.test_event_count,
            "unseen_test_variants": (
                split.unseen_test_variant_count
            ),
        }
    )

split_table = pd.DataFrame(split_rows)
split_table

,object_type,components,train_components,test_components,train_cases,test_cases,train_ratio,train_events,test_events,unseen_test_variants
0,orders,2000,1600,400,1600,400,0.800000,5248,1318,0
1,items,68,44,24,6083,1576,0.794229,16547,4461,28
2,packages,1128,902,226,902,226,0.799645,2950,745,0


## Conformance delle tracce non osservate

Per ogni tipo di oggetto, la rete viene scoperta soltanto dalle tracce di training. Il token-based replay viene quindi eseguito sia sul training set sia sul test set.

Una traccia è considerata fitting se raggiunge il marking finale senza token mancanti o residui.

In [5]:
fitness_rows = []
for evaluation in evaluations:
    train = evaluation.train_conformance
    test = evaluation.test_conformance
    fitness_rows.append(
        {
            "object_type": evaluation.object_type,
            "train_traces": train.trace_count,
            "train_fitting": train.fitting_trace_count,
            "train_fitness": train.average_trace_fitness,
            "test_traces": test.trace_count,
            "test_fitting": test.fitting_trace_count,
            "test_percentage": test.fitting_percentage,
            "test_fitness": test.average_trace_fitness,
            "missing_tokens": test.total_missing_tokens,
            "remaining_tokens": test.total_remaining_tokens,
        }
    )

fitness_table = pd.DataFrame(fitness_rows)
fitness_table

,object_type,train_traces,train_fitting,train_fitness,test_traces,test_fitting,test_percentage,test_fitness,missing_tokens,remaining_tokens
0,orders,1600,1600,1.0,400,400,100.0,1.0,0,0
1,items,6083,6083,1.0,1576,1576,100.0,1.0,0,0
2,packages,902,902,1.0,226,226,100.0,1.0,0,0


## Qualità dei modelli scoperti

La fitness misura quanto comportamento osservato può essere riprodotto. La precisione valuta invece quanto il modello eviti di ammettere comportamento aggiuntivo. La generalizzazione misura la capacità di rappresentare comportamento plausibile non identico alle tracce di training.

La semplicità dipende dalla struttura della Petri net e non dal particolare insieme di tracce sottoposto a replay.

In [6]:
quality_rows = []
for evaluation in evaluations:
    quality_rows.append(
        {
            "object_type": evaluation.object_type,
            "places": evaluation.place_count,
            "transitions": evaluation.transition_count,
            "arcs": evaluation.arc_count,
            "train_precision": evaluation.train_precision,
            "test_precision": evaluation.test_precision,
            "train_generalization": (
                evaluation.train_generalization
            ),
            "test_generalization": (
                evaluation.test_generalization
            ),
            "simplicity": evaluation.model_simplicity,
        }
    )

quality_table = pd.DataFrame(quality_rows)
quality_table

,object_type,places,transitions,arcs,train_precision,test_precision,train_generalization,test_generalization,simplicity
0,orders,7,8,16,0.999589,0.997821,0.955240,0.911296,0.882353
1,items,27,25,64,0.482436,0.472023,0.977151,0.956660,0.684211
2,packages,7,8,16,0.999268,0.998073,0.941271,0.886330,0.882353


## Verifiche automatiche

Le asserzioni controllano l'assenza di identificatori condivisi, la conformità completa delle tracce di test e l'assenza di token mancanti o residui.

In [7]:
total_test_traces = 0
total_fitting_traces = 0

for evaluation in evaluations:
    split = evaluation.split
    test = evaluation.test_conformance

    train_cases = set(
        split.train_log[CASE_COLUMN].astype(str)
    )
    test_cases = set(
        split.test_log[CASE_COLUMN].astype(str)
    )
    train_events = set(
        split.train_log[EVENT_COLUMN].astype(str)
    )
    test_events = set(
        split.test_log[EVENT_COLUMN].astype(str)
    )

    assert train_cases.isdisjoint(test_cases)
    assert train_events.isdisjoint(test_events)
    assert test.non_fitting_trace_count == 0
    assert test.average_trace_fitness == 1.0
    assert test.total_missing_tokens == 0
    assert test.total_remaining_tokens == 0

    total_test_traces += test.trace_count
    total_fitting_traces += test.fitting_trace_count

assert total_test_traces == 2202
assert total_fitting_traces == 2202

print(f"Test traces: {total_test_traces}")
print(f"Fitting traces: {total_fitting_traces}")
print("Leakage-free validation: PASSED")

Test traces: 2202
Fitting traces: 2202
Leakage-free validation: PASSED


## Interpretazione e limiti

Tutte le `2202` tracce out-of-sample risultano fitting, con fitness media pari a `1.0` e senza token mancanti o residui.

Le componenti `orders` e `packages` presentano anche una precisione superiore a `0.99`. La componente `items`, invece, ha precisione pari a circa `0.47`: il modello riproduce tutte le tracce osservate, comprese varianti non presenti nel training, ma ammette anche una quantità rilevante di comportamento aggiuntivo. Questo risultato conferma quantitativamente la struttura ciclica e generalizzante già osservata nella rete degli articoli.

Lo split degli articoli è privo di leakage, ma non è un holdout futuro in senso stretto: alcune componenti connesse attraversano intervalli temporali sovrapposti e devono rimanere indivise per non condividere eventi.

Le soglie e il modello non devono essere selezionati in base ai risultati del test set. Infine, le metriche riguardano Petri net separate per tipo di oggetto e non dimostrano una conformance object-centric globalmente sincronizzata.